# Phase 0 pilot: configuration drift × distribution shift in an open code LLM

Companion to the research proposal *Lossless on the Benchmark, Lossy in Deployment? Reliability of LLM-Based Code Tools under Configuration Drift and Distribution Shift* (Ashhar Hussain).

**What it measures.** Qwen2.5-Coder-1.5B (base) completes all 164 HumanEval tasks with greedy decoding under three numeric configurations: FP16 (reference C0), 8-bit (LLM.int8) and 4-bit NF4. Every configuration runs on the original prompts and on prompts whose function name is anonymised. The rename touches only code tokens and doctest calls, and a task is kept only if its reference solution still passes. C0 is repeated under two other batch compositions to measure the noise floor: outcome flips caused by batching and numerics alone.

**Analysis.** For each task the notebook records pass or fail and compares configurations pairwise against C0. It reports pass@1 with Wilson intervals, loss and gain flips, exact McNemar tests with Holm correction, paired bootstrap intervals, the configuration × shift interaction, and a power analysis for the main study.

**How to run.** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. The full run takes roughly 1 hour on a free T4; set `QUICK = True` for a 40-task check (about 15 minutes). Finished runs are saved, so after a disconnect, *Run all* skips them. Set `SAVE_TO_DRIVE = True` to keep results in Google Drive.

**Safety.** The notebook executes model-generated code. Run it only in Colab's disposable VM, not on your own machine.

In [ ]:
import os, sys, json, time, random, gzip, io, re, hashlib, platform, datetime, subprocess, tempfile, shutil, urllib.request
IN_COLAB = "google.colab" in sys.modules
MOCK_MODEL = os.environ.get("PILOT_MOCK") == "1"   # testing only: a simulated model, no GPU

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B"   # base model, completion-style prompts
QUICK = False                          # True = first 40 tasks only, to check the pipeline
N_TASKS = 40 if QUICK else 164
MAX_NEW_TOKENS = 384
BATCH_SIZE = 16
NOISE_REPEATS = [(8, 1), (12, 2)]       # extra C0 runs: (batch size, shuffle seed)
CONFIGS = ["fp16", "int8", "nf4"]
VARIANTS = ["original", "anon"]
EXEC_TIMEOUT = 10                      # seconds per test program
ALPHA = 0.05
SAVE_TO_DRIVE = False

if MOCK_MODEL:
    N_TASKS = int(os.environ.get("PILOT_N", N_TASKS))
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/llm4se_pilot"
else:
    OUT_DIR = "/content/llm4se_pilot" if IN_COLAB else os.path.abspath("llm4se_pilot")
if MOCK_MODEL:
    OUT_DIR += "_mock"
os.makedirs(OUT_DIR, exist_ok=True)
print("results ->", OUT_DIR, "| tasks:", N_TASKS, "| mock:", MOCK_MODEL)

In [ ]:
# Environment record (library versions are themselves an experimental factor, so they are logged)
if not MOCK_MODEL:
    try:
        import bitsandbytes  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
    import torch, transformers, bitsandbytes
    assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > T4 GPU and run again."

env = {"started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
       "python": platform.python_version(), "model_id": MODEL_ID, "max_new_tokens": MAX_NEW_TOKENS,
       "batch_size": BATCH_SIZE, "noise_repeats": NOISE_REPEATS, "mock_model": MOCK_MODEL}
if not MOCK_MODEL:
    env.update(torch=torch.__version__, transformers=transformers.__version__, bitsandbytes=bitsandbytes.__version__,
               cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0))
with open(os.path.join(OUT_DIR, "pip_freeze.txt"), "w") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
print(json.dumps(env, indent=1))

In [ ]:
# HumanEval (164 tasks) from the official repository
HE_URL = "https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz"
raw = urllib.request.urlopen(HE_URL, timeout=60).read()
ALL = [json.loads(l) for l in gzip.decompress(raw).decode("utf-8").splitlines() if l.strip()]
assert len(ALL) == 164, len(ALL)
env["humaneval_sha256"] = hashlib.sha256(raw).hexdigest()
PROBLEMS = ALL[:N_TASKS]
print(len(PROBLEMS), "tasks | sha256", env["humaneval_sha256"][:16])

In [ ]:
# Execution harness: each program runs in a fresh Python process with a time and memory limit
from concurrent.futures import ThreadPoolExecutor
PREAMBLE = "import resource as _r\ntry:\n    _r.setrlimit(_r.RLIMIT_AS, (2 * 1024 ** 3, 2 * 1024 ** 3))\nexcept Exception:\n    pass\n"

def run_program(program, timeout=EXEC_TIMEOUT):
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "prog.py")
        with open(path, "w") as f:
            f.write(PREAMBLE + program)
        try:
            return subprocess.run([sys.executable, path], cwd=d, capture_output=True, timeout=timeout).returncode == 0
        except subprocess.TimeoutExpired:
            return False

def build_program(prompt, completion, test, entry_point):
    return f"{prompt}{completion}\n\n{test}\n\ncheck({entry_point})\n"

def run_all(programs, workers=8):
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(ex.map(run_program, programs))

canon_ok = run_all([build_program(p["prompt"], p["canonical_solution"], p["test"], p["entry_point"]) for p in PROBLEMS])
print(f"reference solutions passing: {sum(canon_ok)}/{len(canon_ok)}")
assert sum(canon_ok) >= len(canon_ok) - 1, "execution harness is not working as expected"

In [ ]:
# Shift condition: anonymise the function name (code tokens + doctest calls only; prose in docstrings is untouched)
import tokenize
NEW_NAME = "task_func"

def rename_identifier(src, old, new):
    starts = [0]
    for line in src.splitlines(keepends=True):
        starts.append(starts[-1] + len(line))
    pos = lambda rc: starts[rc[0] - 1] + rc[1]
    edits = []
    for tok in tokenize.generate_tokens(io.StringIO(src).readline):
        if tok.type == tokenize.NAME and tok.string == old:
            edits.append((pos(tok.start), pos(tok.end)))
        elif tok.type == tokenize.STRING:
            base = pos(tok.start)
            edits += [(base + m.start(), base + m.end()) for m in re.finditer(rf"\b{re.escape(old)}\b(?=\s*\()", tok.string)]
    out = src
    for a, b in sorted(edits, reverse=True):
        out = out[:a] + new + out[b:]
    return out, edits

def anonymise(p):
    if re.search(rf"\b{NEW_NAME}\b", p["prompt"] + p["canonical_solution"] + p["test"]):
        return None
    full, cut = p["prompt"] + p["canonical_solution"], len(p["prompt"])
    new_full, edits = rename_identifier(full, p["entry_point"], NEW_NAME)
    assert all(b <= cut or a >= cut for a, b in edits)
    new_cut = cut + sum(len(NEW_NAME) - (b - a) for a, b in edits if b <= cut)
    return dict(p, prompt=new_full[:new_cut], canonical_solution=new_full[new_cut:],
                test=rename_identifier(p["test"], p["entry_point"], NEW_NAME)[0], entry_point=NEW_NAME)

ANON = {p["task_id"]: q for p in PROBLEMS if (q := anonymise(p)) is not None}
ok = run_all([build_program(q["prompt"], q["canonical_solution"], q["test"], q["entry_point"]) for q in ANON.values()])
ANON = {tid: q for (tid, q), good in zip(ANON.items(), ok) if good}
TASKS = [p for p, good in zip(PROBLEMS, canon_ok) if good and p["task_id"] in ANON]
variant_problem = lambda p, v: p if v == "original" else ANON[p["task_id"]]
env["tasks_used"] = len(TASKS)
print(f"{len(TASKS)} tasks usable in both conditions. Example anonymised prompt:\n")
print(ANON[TASKS[0]["task_id"]]["prompt"])

In [ ]:
# Generation: greedy decoding, standard HumanEval stop sequences, left padding
STOP = ["\nclass", "\ndef", "\n#", "\nif", "\nprint", "\nassert"]

def truncate(text):
    cuts = [i for i in (text.find(s) for s in STOP) if i != -1]
    return text[:min(cuts)] if cuts else text

if not MOCK_MODEL:
    import gc
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList
    TOK = AutoTokenizer.from_pretrained(MODEL_ID)
    TOK.padding_side = "left"
    if TOK.pad_token is None:
        TOK.pad_token = TOK.eos_token

    class StopOnStrings(StoppingCriteria):
        def __init__(self, start):
            self.start = start
        def __call__(self, input_ids, scores, **kwargs):
            texts = TOK.batch_decode(input_ids[:, self.start:], skip_special_tokens=True)
            return torch.tensor([any(s in t for s in STOP) for t in texts], dtype=torch.bool, device=input_ids.device)

    def load_model(cfg):
        kw = {"device_map": {"": 0}, "torch_dtype": torch.float32 if cfg == "fp32" else torch.float16}
        if cfg == "int8":
            kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        elif cfg == "nf4":
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16)
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw).eval()
        env.setdefault("model_revision", getattr(m.config, "_commit_hash", None))
        return m

    @torch.no_grad()
    def logits_finite(m):
        enc = TOK([TASKS[0]["prompt"]], return_tensors="pt").to(m.device)
        return bool(torch.isfinite(m(**enc).logits).all())

    @torch.no_grad()
    def generate(m, prompts, batch_size, order_seed=None):
        order = list(range(len(prompts)))
        if order_seed is not None:
            random.Random(order_seed).shuffle(order)
        out = [None] * len(prompts)
        for i in range(0, len(order), batch_size):
            batch = order[i:i + batch_size]
            enc = TOK([prompts[j] for j in batch], return_tensors="pt", padding=True).to(m.device)
            start = enc["input_ids"].shape[1]
            gen = m.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=TOK.pad_token_id,
                             stopping_criteria=StoppingCriteriaList([StopOnStrings(start)]))
            for j, text in zip(batch, TOK.batch_decode(gen[:, start:], skip_special_tokens=True)):
                out[j] = truncate(text)
        return out
else:
    MOCK_RATE = {"fp16": 0.45, "int8": 0.44, "nf4": 0.40}
    def u(*key):
        return int(hashlib.sha256("|".join(map(str, key)).encode()).hexdigest(), 16) % 10_000 / 10_000
    def generate_mock(variant, cfg, seed):
        rate = MOCK_RATE[cfg] - (0.04 if variant == "anon" else 0.0)
        return [variant_problem(p, variant)["canonical_solution"]
                if 0.85 * u(p["task_id"]) + 0.15 * u(p["task_id"], variant, cfg, seed) < rate else "    return None\n"
                for p in TASKS]

In [ ]:
# Run every configuration x condition; finished runs are skipped on re-run
RUNS = [("C0", "fp16", BATCH_SIZE, None)]
RUNS += [(f"C0_rep{k + 1}", "fp16", bs, seed) for k, (bs, seed) in enumerate(NOISE_REPEATS)]
RUNS += [(cfg, cfg, BATCH_SIZE, None) for cfg in CONFIGS if cfg != "fp16"]
result_path = lambda run, v: os.path.join(OUT_DIR, f"{run}__{v}.jsonl")

def save(run, cfg, bs, seed, v, completions, seconds, peak_gb):
    probs = [variant_problem(p, v) for p in TASKS]
    passed = run_all([build_program(q["prompt"], c, q["test"], q["entry_point"]) for q, c in zip(probs, completions)])
    with open(result_path(run, v), "w") as f:
        for p, c, ok in zip(TASKS, completions, passed):
            f.write(json.dumps({"task_id": p["task_id"], "run": run, "config": cfg, "variant": v, "passed": bool(ok), "completion": c}) + "\n")
    meta = {"run": run, "config": cfg, "variant": v, "batch_size": bs, "order_seed": seed, "seconds": round(seconds, 1),
            "peak_gpu_gb": peak_gb, "pass_at_1": round(sum(passed) / len(passed), 4)}
    with open(os.path.join(OUT_DIR, "runs_meta.jsonl"), "a") as f:
        f.write(json.dumps(meta) + "\n")
    print(f"{run:>8} | {v:>8} | pass@1 {meta['pass_at_1']:.3f} | {meta['seconds']:.0f} s | peak {peak_gb} GB")

model, loaded = None, None
for run, cfg, bs, seed in RUNS:
    todo = [v for v in VARIANTS if not os.path.exists(result_path(run, v))]
    if not todo:
        print(f"{run}: done earlier, skipped")
        continue
    if MOCK_MODEL:
        for v in todo:
            save(run, cfg, bs, seed, v, generate_mock(v, cfg, seed), 0.0, None)
        continue
    if loaded != cfg:
        if model is not None:
            del model; gc.collect(); torch.cuda.empty_cache()
        model, loaded = load_model(cfg), cfg
        if cfg == "fp16" and not logits_finite(model):
            print("FP16 logits are not finite for this model; FP32 is used as the reference instead.")
            env["reference_dtype"] = "fp32 (fp16 logits not finite)"
            del model; gc.collect(); torch.cuda.empty_cache()
            model = load_model("fp32")
    for v in todo:
        torch.cuda.reset_peak_memory_stats(); t0 = time.time()
        comps = generate(model, [variant_problem(p, v)["prompt"] for p in TASKS], bs, seed)
        save(run, cfg, bs, seed, v, comps, time.time() - t0, round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2))
env.setdefault("reference_dtype", "fp16")
env["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
with open(os.path.join(OUT_DIR, "environment.json"), "w") as f:
    json.dump(env, f, indent=1)

In [ ]:
# Statistics: Wilson intervals, exact McNemar, Holm, paired bootstrap
import numpy as np, pandas as pd
from scipy.stats import binom

def wilson(k, n, z=1.959963984540054):
    p, den = k / n, 1 + z * z / n
    c, h = (p + z * z / (2 * n)) / den, z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

def mcnemar_exact(b, c):
    return 1.0 if b + c == 0 else float(min(1.0, 2 * binom.cdf(min(b, c), b + c, 0.5)))

def holm(p):
    p = np.asarray(p, float); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(np.argsort(p)):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj

def boot_mean(x, B=10000, seed=0):
    x = np.asarray(x, float); idx = np.random.default_rng(seed).integers(0, len(x), (B, len(x)))
    lo, hi = np.percentile(x[idx].mean(1), [2.5, 97.5])
    return x.mean(), lo, hi

rows = []
for run, *_ in RUNS:
    for v in VARIANTS:
        with open(result_path(run, v)) as f:
            rows += [json.loads(l) for l in f]
P = pd.DataFrame(rows).pivot_table(index="task_id", columns=["run", "variant"], values="passed", aggfunc="first").dropna().astype(int)
n = len(P)
flips = lambda a, b: (int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum()))

pass_tbl = pd.DataFrame([{"run": r, "config": c, "variant": v, "pass@1": P[(r, v)].mean(),
                          "wilson_lo": wilson(P[(r, v)].sum(), n)[0], "wilson_hi": wilson(P[(r, v)].sum(), n)[1]}
                         for r, c, *_ in RUNS for v in VARIANTS])

c0_runs = [r for r, *_ in RUNS if r.startswith("C0")]
noise = pd.DataFrame([{"variant": v, "pair": f"{a} vs {b}", "a_pass_b_fail": f[0], "a_fail_b_pass": f[1]}
                      for v in VARIANTS for i, a in enumerate(c0_runs) for b in c0_runs[i + 1:]
                      for f in [flips(P[(a, v)], P[(b, v)])]])
NOISE_MAX = {v: int(noise.loc[noise.variant == v, ["a_pass_b_fail", "a_fail_b_pass"]].values.max()) for v in VARIANTS}

eff = []
for r, c, *_ in RUNS:
    if r.startswith("C0"):
        continue
    for v in VARIANTS:
        a, b = P[("C0", v)].values, P[(r, v)].values
        lo_, gain = flips(a, b); m, l, h = boot_mean(b - a)
        eff.append({"config": c, "variant": v, "losses": lo_, "gains": gain, "delta_pass@1": m, "ci_lo": l, "ci_hi": h,
                    "p_mcnemar": mcnemar_exact(lo_, gain), "noise_floor": NOISE_MAX[v]})
eff = pd.DataFrame(eff)
eff["p_holm"] = holm(eff["p_mcnemar"])
eff["harmful_effect_claimed"] = (eff["p_holm"] < ALPHA) & (eff["losses"] > eff["noise_floor"])

shift = []
for r, c, *_ in RUNS:
    if r.startswith("C0_rep"):
        continue
    a, b = P[(r, "original")].values, P[(r, "anon")].values
    lo_, gain = flips(a, b); m, l, h = boot_mean(b - a)
    shift.append({"config": c, "losses": lo_, "gains": gain, "delta_pass@1": m, "ci_lo": l, "ci_hi": h, "p_mcnemar": mcnemar_exact(lo_, gain)})
shift = pd.DataFrame(shift); shift["p_holm"] = holm(shift["p_mcnemar"])

inter = []
for r, c, *_ in RUNS:
    if r.startswith("C0"):
        continue
    d_o = P[(r, "original")].values - P[("C0", "original")].values
    d_a = P[(r, "anon")].values - P[("C0", "anon")].values
    m, l, h = boot_mean(d_a - d_o)
    inter.append({"config": c, "effect_original": d_o.mean(), "effect_anon": d_a.mean(), "difference": m, "ci_lo": l, "ci_hi": h})
inter = pd.DataFrame(inter)

pd.set_option("display.float_format", "{:.3f}".format)
print(f"paired tasks: {n}\n\nPASS@1\n", pass_tbl.to_string(index=False))
print("\nNOISE FLOOR (flips between repeated C0 runs)\n", noise.to_string(index=False))
print("\nCONFIGURATION EFFECTS vs C0\n", eff.to_string(index=False))
print("\nSHIFT EFFECTS (anonymised minus original)\n", shift.to_string(index=False))
print("\nINTERACTION (config effect under shift minus config effect on original)\n", inter.to_string(index=False))

In [ ]:
# Power analysis for the main study: tasks needed to detect each observed asymmetry (exact McNemar, Bonferroni alpha)
def power(p10, p01, n_tasks, alpha, sims=4000, seed=0):
    pd_ = p10 + p01
    if pd_ == 0 or p10 == p01:
        return float("nan")
    rng = np.random.default_rng(seed)
    D = rng.binomial(n_tasks, pd_, sims); Bc = rng.binomial(D, p10 / pd_)
    k = np.minimum(Bc, D - Bc)
    p = np.where(D > 0, np.minimum(1.0, 2 * binom.cdf(k, np.maximum(D, 1), 0.5)), 1.0)
    return float((p < alpha).mean())

GRID = [164, 300, 500, 1000, 2000, 5000]
a_adj = ALPHA / len(eff)
pw = []
for _, e in eff.iterrows():
    vals = {g: power(e.losses / n, e.gains / n, g, a_adj) for g in GRID}
    need = next((g for g in GRID if vals[g] == vals[g] and vals[g] >= 0.8), None)
    pw.append({"config": e.config, "variant": e.variant, **{f"power@{g}": vals[g] for g in GRID},
               "tasks_for_80%_power": need if need else ("no asymmetry observed" if e.losses == e.gains else "> 5000")})
pw = pd.DataFrame(pw)
print(pw.to_string(index=False))

In [ ]:
# Figures and export
import matplotlib.pyplot as plt
BLUE, ORANGE = "#2a5c9a", "#d17a22"
order = [r for r, *_ in RUNS]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
x = np.arange(len(order)); w = 0.38
for k, (v, col) in enumerate([("original", BLUE), ("anon", ORANGE)]):
    t = pass_tbl[pass_tbl.variant == v].set_index("run").loc[order]
    axes[0].bar(x + (k - 0.5) * w, t["pass@1"], w - 0.03, color=col, label=v,
                yerr=[t["pass@1"] - t["wilson_lo"], t["wilson_hi"] - t["pass@1"]], capsize=3, ecolor="#555")
axes[0].set_xticks(x, order); axes[0].set_ylabel("pass@1 (95% Wilson CI)"); axes[0].set_title("Pass@1 by run and prompt condition")
axes[0].legend(frameon=False)
labels = [f"{c}\n{v}" for c, v in zip(eff.config, eff.variant)]
xx = np.arange(len(eff))
axes[1].bar(xx - 0.2, eff.losses, 0.38, color=BLUE, label="losses (C0 pass, config fail)")
axes[1].bar(xx + 0.2, eff.gains, 0.38, color=ORANGE, label="gains (C0 fail, config pass)")
axes[1].scatter(xx, eff.noise_floor, marker="_", s=900, color="#222", label="noise floor (max C0 repeat flips)", zorder=3)
axes[1].set_xticks(xx, labels); axes[1].set_ylabel("tasks"); axes[1].set_title("Outcome flips vs reference C0")
axes[1].legend(frameon=False, fontsize=8)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "pilot_figure.png"), dpi=200); plt.show()

for name, t in [("pass_at_1", pass_tbl), ("noise_floor", noise), ("config_effects", eff), ("shift_effects", shift),
                ("interaction", inter), ("power", pw)]:
    t.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=False)
archive = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("saved:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)

## Reading the results

* A configuration has a **harmful effect** only if its Holm-adjusted McNemar test rejects **and** its loss flips exceed the noise floor measured between repeated C0 runs. A difference in pass@1 alone is not evidence.
* **Interaction** above zero means that the configuration's effect is more favourable on anonymised prompts than on original ones; below zero means the configuration hurts more under shift. The bootstrap interval shows whether that difference can be told apart from zero.
* The **power table** gives the number of tasks the main study needs to detect each observed asymmetry with 80% power. That number decides how much benchmark data (HumanEval+, MBPP+, LiveCodeBench post-cutoff slices) the main study has to use.

**Limits of this pilot:** one 1.5B model, greedy decoding, one GPU type, one perturbation, and HumanEval, which is likely present in the model's training data. Report only numbers that this notebook actually produced, together with `environment.json`.